# Notebook 05 — Generating synthetic reviews with Claude

The TripAdvisor dataset doesn't include review text, only aggregate statistics.
This notebook fills the gap by asking Claude to generate **plausible synthetic 
reviews** for each restaurant, grounded in the actual restaurant metadata 
(cuisine, price, rating, vibes, signatures, shortcomings).

This is **transparently synthetic**: a note in the README will state that 
reviews are LLM-generated based on factual restaurant data. The reviews are 
designed to be coherent with each restaurant's actual profile.

We generate **2 reviews per restaurant** (one positive-leaning, one critical-leaning) 
to give Claude variety when the agent picks a review to show.

**Estimated cost**: ~0.80 USD for 40 restaurants × 2 reviews.

In [1]:
"""Notebook 05 — Generate synthetic user reviews for each enriched restaurant."""

import json
import os
import random
from datetime import datetime, timedelta
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from tqdm import tqdm

# Setup
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INPUT_JSON = DATA_PROCESSED / "lyon_restaurants_with_vibes.json"
OUTPUT_JSON = DATA_PROCESSED / "lyon_reviews.json"

# Anthropic client
client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 1024

# Number of reviews to generate per restaurant
REVIEWS_PER_RESTAURANT = 2

# Seed for reproducibility (so re-runs produce same dates/IDs)
random.seed(42)

print(f"API key loaded: {bool(os.environ.get('ANTHROPIC_API_KEY'))}")

API key loaded: True


In [2]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    restaurants = json.load(f)

print(f"Loaded {len(restaurants)} enriched restaurants")
print(f"Sample restaurant fields: {list(restaurants[0].keys())}")

Loaded 148 enriched restaurants
Sample restaurant fields: ['restaurant_link', 'restaurant_name', 'primary_cuisine', 'cuisines', 'price_level', 'avg_rating', 'total_reviews_count', 'address', 'vibes', 'description', 'signatures', 'shortcomings']


In [3]:
SYSTEM_PROMPT = """You are a TripAdvisor user writing an honest, natural review \
of a Lyon restaurant. Your tone matches everyday users — not a food critic, \
not a marketing agent.

Style guidelines:
- Conversational, slightly personal (mention companion, occasion, mood)
- Specific details that prove you actually went there
- Honest: include what disappointed even in positive reviews
- 100-200 words, NOT longer
- English (since this is the international TripAdvisor version)
- Mention specific dishes when relevant (use French dish names)

CRITICAL: Output STRICTLY valid JSON with these exact keys:
{
  "title": "short review title (5-10 words)",
  "text": "the review body (100-200 words)",
  "rating": <number between 1.0 and 5.0, in 0.5 increments>,
  "userId": "USER_<descriptive_handle_in_caps>"
}

No markdown, no preamble, just the JSON."""


def build_review_prompt(restaurant: dict, lean: str) -> str:
    """
    Build a review prompt for one restaurant.
    
    lean: 'positive' for a 4.0-5.0 rating, 'critical' for a 3.0-4.0 rating
    """
    rating_guidance = {
        "positive": "Write an overall POSITIVE review (rating 4.0 to 5.0). Mention 1 small thing that could be improved for honesty.",
        "critical": "Write a MIXED review (rating 3.0 to 4.0). The visit was okay but not great. Be specific about what fell flat.",
    }[lean]
    
    return f"""Restaurant to review:
- Name: {restaurant['restaurant_name']}
- Cuisine: {restaurant.get('primary_cuisine', 'Unknown')}
- Price level: {restaurant.get('price_level', '?')}
- Overall rating on TripAdvisor: {restaurant.get('avg_rating', '?')}/5 ({restaurant.get('total_reviews_count', '?')} reviews)
- Atmosphere vibes: {', '.join(restaurant.get('vibes', []))}
- Likely signature dishes: {', '.join(restaurant.get('signatures', []))}
- Known shortcomings: {' | '.join(restaurant.get('shortcomings', []))}
- Description: {restaurant.get('description', '')}

{rating_guidance}

Output the review as JSON with title, text, rating, and userId. Output ONLY the JSON."""

In [4]:
test_restaurant = restaurants[0]
test_prompt = build_review_prompt(test_restaurant, lean="positive")

print(f"Generating review for: {test_restaurant['restaurant_name']}")
print("=" * 70)

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": test_prompt}],
)

text = response.content[0].text.strip()

# Defensive markdown strip (we learned this in Session 5!)
if text.startswith("```"):
    text = text.split("\n", 1)[1] if "\n" in text else text
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()

review = json.loads(text)

print(f"\n📝 Title: {review['title']}")
print(f"⭐ Rating: {review['rating']}")
print(f"👤 User: {review['userId']}")
print(f"\n💬 Review:\n{review['text']}")
print(f"\nTokens: input={response.usage.input_tokens}, output={response.usage.output_tokens}")

Generating review for: Lyon-Dakar

📝 Title: Authentic West African food, worth the wait
⭐ Rating: 4.5
👤 User: USER_LYON_FOOD_EXPLORER

💬 Review:
Took my partner here on a Saturday evening and we were genuinely impressed. The thiéboudienne was excellent—properly seasoned rice with tender fish, exactly what it should be. We also ordered the grilled lamb with sauce d'arachide, which had real depth and wasn't overly heavy despite the groundnut base.

The atmosphere is exactly as advertised: casual, lively, and full of locals who clearly know what they're doing. No fancy plating, no pretence—just good food prepared with care. Prices are fair for the quality and portion sizes.

My only complaint: service was quite slow on Saturday night. We waited ages between ordering and eating, and our water glasses weren't refilled often. That said, I got the sense the kitchen was the priority here, not turning tables quickly, which is honestly refreshing. The wine list is basic, but that's fine—I just g

In [5]:
def generate_one_review(restaurant: dict, lean: str, max_retries: int = 2) -> dict | None:
    """Generate one review for a restaurant. Returns the parsed dict or None."""
    prompt = build_review_prompt(restaurant, lean=lean)
    
    for attempt in range(max_retries + 1):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )
            text = response.content[0].text.strip()
            
            # Strip markdown fences defensively
            if text.startswith("```"):
                text = text.split("\n", 1)[1] if "\n" in text else text
                if text.endswith("```"):
                    text = text[:-3]
                text = text.strip()
            
            return json.loads(text)
        
        except json.JSONDecodeError as e:
            if attempt < max_retries:
                continue
            print(f"  ❌ JSON parse failure: {e}")
            return None
        
        except Exception as e:
            print(f"  ❌ API error: {e}")
            return None
    
    return None


def random_visit_date() -> str:
    """Generate a random date in the last 18 months."""
    today = datetime(2026, 6, 1)
    days_ago = random.randint(0, 540)  # 0 to ~18 months ago
    visit = today - timedelta(days=days_ago)
    return visit.strftime("%Y-%m-%d")


def make_review_id(idx: int) -> int:
    """Generate a stable review ID."""
    return 900000000 + idx

In [6]:
if OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        all_reviews = json.load(f)
    done_pairs = {(r["itemId"], r.get("lean", "?")) for r in all_reviews}
    print(f"Resuming: {len(all_reviews)} reviews already generated")
else:
    all_reviews = []
    done_pairs = set()
    print("Starting from scratch")

Starting from scratch


In [7]:
# We need an itemId for each restaurant — use the restaurant_link as basis
# but convert to numeric IDs for cleanliness.
for idx, r in enumerate(restaurants):
    r["itemId"] = 1000000 + idx  # synthetic itemId

leans = ["positive", "critical"]
total_to_generate = len(restaurants) * len(leans)

with tqdm(total=total_to_generate, desc="Generating reviews") as pbar:
    review_idx = len(all_reviews)
    
    for restaurant in restaurants:
        for lean in leans:
            # Skip if already done
            if (restaurant["itemId"], lean) in done_pairs:
                pbar.update(1)
                continue
            
            review_data = generate_one_review(restaurant, lean=lean)
            if review_data is None:
                pbar.update(1)
                continue
            
            # Build the full record matching the server's expected schema
            record = {
                "reviewId": make_review_id(review_idx),
                "userId": review_data.get("userId", "USER_ANONYMOUS"),
                "itemId": restaurant["itemId"],
                "title": review_data.get("title", ""),
                "text": review_data.get("text", ""),
                "date": random_visit_date(),
                "rating": review_data.get("rating", 4.0),
                "language": "en",
                "lean": lean,  # bookkeeping
            }
            all_reviews.append(record)
            review_idx += 1
            
            # Save after each successful generation
            with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
                json.dump(all_reviews, f, indent=2, ensure_ascii=False)
            
            pbar.update(1)

print(f"\n✅ Done. {len(all_reviews)} reviews generated.")

Generating reviews: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 296/296 [20:16<00:00,  4.11s/it]


✅ Done. 296 reviews generated.


In [8]:
sample = random.sample(all_reviews, min(4, len(all_reviews)))

for r in sample:
    # Find the matching restaurant
    matching_restaurant = next(
        (rest for rest in restaurants if rest["itemId"] == r["itemId"]),
        None,
    )
    rest_name = matching_restaurant["restaurant_name"] if matching_restaurant else "?"
    
    print(f"\n{'=' * 70}")
    print(f"🍽  {rest_name}")
    print(f"   Lean: {r['lean']}")
    print(f"\n📝 {r['title']}")
    print(f"   👤 {r['userId']} · ⭐ {r['rating']}/5 · 📅 {r['date']}")
    print(f"\n{r['text']}")


🍽  Maison Rousseau
   Lean: positive

📝 Fresh seafood in the heart of the market
   👤 USER_LYON_SEAFOOD_LOVER · ⭐ 4.5/5 · 📅 2025-09-30

Took my partner here on a Saturday afternoon and really enjoyed it. The location inside Halles de Lyon Paul Bocuse is brilliant—you can see exactly where your fish is coming from. We ordered the plateau de fruits de mer and sole meunière, both impeccably fresh and simply prepared. The kitchen clearly respects the ingredients rather than hiding behind fancy techniques, which I appreciated. Service was efficient and friendly, fitting the casual-refined vibe. The only downside? It was quite loud during our visit—the market hall atmosphere adds charm but definitely impacts the quietness factor if you're after intimate conversation. Also, prices felt a bit steep when the daily catch is limited, though most days seem well-stocked. Still, for Lyon, this hits the mark perfectly. Would return.

🍽  Sainte Russie
   Lean: positive

📝 Genuine Russian charm in the